# K eep
# I t
# S imple
# S tupid

# 1. DADOS

In [32]:
import pandas as pd
from common.utils import get_project_root

project_root_dir = get_project_root(project_name="classificador-cbo")
silver_layer = project_root_dir / "data/silver"
path_treated_data = silver_layer/  "treated/familia.csv"

treated_data = pd.read_csv(path_treated_data, sep="\t")
treated_data["codigo"] = treated_data["codigo"].astype("category")
treated_data.head()

,codigo,sintese,perfilocupacional,titulo
0,1113,atua processo julgamento causas relacionadas j...,realiza pesquisas estudos área direito áreas a...,Magistrados
1,1114,atua gestor público órgãos administração públi...,atua manter aprimoramento contínuo aspectos qu...,Dirigentes do serviço público
2,1115,coordena supervisiona executa funções relacion...,conduz orienta pesquisas socioeconômicas estud...,Gestores públicos
3,1141,exerce funções dirigente político partidário e...,prepara exercício funções político partidárias...,Dirigentes de partidos políticos
4,1142,exerce funções direção entidade trabalhadores ...,prepara atuar entidade patronal analisando leg...,Dirigentes e administradores de entidades patr...


In [33]:
from sentence_transformers import SentenceTransformer
import joblib
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L12-v2")
count_vect = CountVectorizer()
tfidf_vect = TfidfVectorizer()

# Síntese
cols = [
    "codigo",
    "titulo",
    "sintese"
]
sintese = treated_data.loc[:,cols].copy()

sintese["count_embedding"] = list(count_vect.fit_transform(raw_documents=sintese["sintese"]).toarray())
sintese["tfidf_embedding"] = list(tfidf_vect.fit_transform(raw_documents=sintese["sintese"]).toarray())
sintese["transformer_embedding"] = list(model.encode(sentences=sintese["sintese"]))

path_sintese_embeddings = silver_layer / "embeddings/sintese_familias.joblib"
path_sintese_embeddings.parent.mkdir(exist_ok=True, parents=True)
with open(path_sintese_embeddings, "wb") as joblib_f:
    joblib.dump(sintese, joblib_f)

# Perfil
cols = [
    "codigo",
    "titulo",
    "perfilocupacional"
]
perfil_ocupacional = treated_data.loc[:,cols].copy()

perfil_ocupacional["count_embedding"] = list(count_vect.fit_transform(raw_documents=perfil_ocupacional["perfilocupacional"]).toarray())
perfil_ocupacional["tfidf_embedding"] = list(tfidf_vect.fit_transform(raw_documents=perfil_ocupacional["perfilocupacional"]).toarray())
perfil_ocupacional["transformer_embedding"] = list(model.encode(sentences=perfil_ocupacional["perfilocupacional"]))

path_perfilocupacional_embeddings = silver_layer / "embeddings/perfilocupacional_familias.joblib"
with open(path_perfilocupacional_embeddings, "wb") as joblib_f:
    joblib.dump(perfil_ocupacional, joblib_f)

In [34]:
perfil_ocupacional["target"] = perfil_ocupacional["codigo"].astype(str).str[0].astype("category")
perfil_ocupacional.head()

,codigo,titulo,perfilocupacional,count_embedding,tfidf_embedding,transformer_embedding,target
0,1113,Magistrados,realiza pesquisas estudos área direito áreas a...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.048082553, 0.039967574, -0.041692607, -0.02...",1
1,1114,Dirigentes do serviço público,atua manter aprimoramento contínuo aspectos qu...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.024086827, 0.020039512, -0.0016491849, -0.0...",1
2,1115,Gestores públicos,conduz orienta pesquisas socioeconômicas estud...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.036369786, -0.07156354, -0.020440344, -0.00...",1
3,1141,Dirigentes de partidos políticos,prepara exercício funções político partidárias...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.006765382, 0.0441265, 0.006600018, -0.02568...",1
4,1142,Dirigentes e administradores de entidades patr...,prepara atuar entidade patronal analisando leg...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.075171925, -0.012606906, 0.02053122, -0.016...",1


# 2. TREINAMENTO DO MODELO

In [38]:
perfil_ocupacional.iloc[0,-4].shape

(24628,)

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=len(treated_data)
)

# 3. USO DO MODELO